# Run 3 — More Facts + Supplier Update + Backdated Dimension Correction (`batch_id = 3`)

- **Fact append:** another batch of new orders + line items grows `fct_order_lines` further.
- **Supplier update (effective 1997-01-01):** a couple of suppliers get a new SCD2 version, giving
  them multi-version history across Run 2 (1996) and Run 3 (1997) on the business timeline.
- **Backdated dimension correction (effective 1994-01-01):** a late-arriving `customer_address`
  correction that arrives now but is effective *before* the Run 2 (1996) update. Silver
  `sequence_by: effective_date` slots it into history between the 1992 baseline and the 1996 update
  while the 1996 value stays current — demonstrating correct out-of-order handling on the business
  timeline.
- **Supplier delete (effective 1997-01-01):** supplier 3 is discontinued and arrives with
  `cdc_operation='D'`; `cdcSettings.apply_as_deletes` closes its open SCD2 version (tombstone). Historical
  orders still resolve it via the as-of join; orders after the delete resolve to the unknown
  member.
- **Late-arriving dimension master:** part `9000001` (referenced by a Run-2 line item that
  resolved to the unknown member) finally arrives. New Run-3 facts for it resolve to the real
  part; the Run-2 fact keeps the unknown member (append-only facts are not retro-repointed).
- **Data-quality quarantine:** a second malformed batch (bad `customer_address` + `orders` rows)
  lands and is routed to the silver `*_quarantine` tables, showing quarantine continuing to catch
  bad data across incremental runs.

In [ ]:
%run ./initialize

In [ ]:
# Fact append: another batch of new orders + line items (different offset / slice)
batch_id = 3
load_ts = datetime.now()

ORDER_OFFSET = 200000000
ORDER_FILTER = "o_orderkey > 60000 AND o_orderkey <= 120000"
LINE_FILTER = "l_orderkey > 60000 AND l_orderkey <= 120000"

new_orders = spark.sql(f"""
    SELECT o_orderkey + {ORDER_OFFSET} AS o_orderkey, o_custkey, o_orderstatus, o_totalprice,
           date_add(o_orderdate, 2200) AS o_orderdate, o_orderpriority, o_clerk, o_shippriority, o_comment
    FROM {sample_source_schema}.orders
    WHERE {ORDER_FILTER}
""")
# DQ demo (run 3): another malformed orders batch -> dropped from the clean fact feed
# and captured in silver.orders_quarantine (quarantineMode=table).
bad_orders = spark.sql(f"""
    SELECT CAST(NULL AS BIGINT) AS o_orderkey, o_custkey, o_orderstatus, o_totalprice,
           date_add(o_orderdate, 2200) AS o_orderdate, o_orderpriority, o_clerk, o_shippriority,
           'DQ_DEMO (run3): null order_key (PK)' AS o_comment
    FROM {sample_source_schema}.orders WHERE o_orderkey = 3
    UNION ALL
    SELECT o_orderkey + 910000000 AS o_orderkey, o_custkey, o_orderstatus,
           CAST(-99999.99 AS DECIMAL(18,2)) AS o_totalprice,
           date_add(o_orderdate, 2200) AS o_orderdate, o_orderpriority, o_clerk, o_shippriority,
           'DQ_DEMO (run3): negative total_price' AS o_comment
    FROM {sample_source_schema}.orders WHERE o_orderkey = 4
""")
new_orders = new_orders.unionByName(bad_orders)
write_landing(with_metadata(new_orders, "order_mgmt", batch_id, load_ts), "order_mgmt", "orders", INCREMENTAL_ZONE, load_ts)

new_lineitems = spark.sql(f"""
    SELECT l_orderkey + {ORDER_OFFSET} AS l_orderkey, l_partkey, l_suppkey, l_linenumber,
           l_quantity, l_extendedprice, l_discount, l_tax, l_returnflag, l_linestatus,
           date_add(l_shipdate, 2200) AS l_shipdate, date_add(l_commitdate, 2200) AS l_commitdate,
           date_add(l_receiptdate, 2200) AS l_receiptdate, l_shipinstruct, l_shipmode, l_comment
    FROM {sample_source_schema}.lineitem
    WHERE {LINE_FILTER}
""")

# Resolution side of the late-arriving dimension demo: a NEW line item for part 9000001, whose
# master record arrives in this same run (see cell below). Unlike the Run-2 fact that resolved to
# the unknown member, this one resolves to the real dim_part member. line_number 99 avoids
# colliding with the order's real lines.
resolved_part_line = spark.sql(f"""
    SELECT l_orderkey + {ORDER_OFFSET} AS l_orderkey, CAST(9000001 AS BIGINT) AS l_partkey, l_suppkey,
           CAST(99 AS INT) AS l_linenumber, l_quantity, l_extendedprice, l_discount, l_tax,
           l_returnflag, l_linestatus, date_add(l_shipdate, 2200) AS l_shipdate,
           date_add(l_commitdate, 2200) AS l_commitdate, date_add(l_receiptdate, 2200) AS l_receiptdate,
           l_shipinstruct, l_shipmode, 'LATE_ARRIVAL_DEMO: part 9000001 now resolvable' AS l_comment
    FROM {sample_source_schema}.lineitem WHERE l_orderkey = 60001 AND l_linenumber = 1
""")
new_lineitems = new_lineitems.unionByName(resolved_part_line)
write_landing(with_metadata(new_lineitems, "order_fulfillment", batch_id, load_ts), "order_fulfillment", "lineitem", INCREMENTAL_ZONE, load_ts)

print("Run 3 fact append staged (batch_id = 3)")

In [ ]:
# Dimension changes (Run 3):
#   1) Supplier update (effective 1997) -> a forward SCD2 version for a couple of suppliers, giving
#      them multi-version history across Run 2 (1996) and Run 3 (1997). Orders resolve the supplier
#      name effective as of the order_date.
#   2) Backdated customer_address correction (effective 1994) for customer 1. It ARRIVES now
#      (Run 3) but is effective BEFORE the Run 2 (1996) update. Because silver sequences SCD2 by
#      effective_date, it slots into history between the 1992 baseline and the 1996 update, while
#      the 1996 value stays current. Orders placed in 1994-1995 resolve to this corrected version
#      -- out-of-order handling on the business timeline (a backdated correction).
RUN3_SUPPLIER_EFFECTIVE = "1997-01-01 00:00:00"
RUN3_CORRECTION_EFFECTIVE = "1994-01-01 00:00:00"

demo_supplier_keys = "(1, 2)"
supp3 = spark.sql(f"SELECT s_suppkey AS supplier_id, concat(s_name, ' (RENAMED-2)') AS name, s_acctbal AS acctbal, s_comment AS comment FROM {sample_source_schema}.supplier WHERE s_suppkey IN {demo_supplier_keys}")
supp3 = with_op(with_effective(with_metadata(supp3, "procurement", batch_id, load_ts), RUN3_SUPPLIER_EFFECTIVE))

# Delete demo: supplier 3 is discontinued. It arrives with cdc_operation='D'; silver applies it
# via cdcSettings.apply_as_deletes ("cdc_operation = 'D'"), which CLOSES the open SCD2 version (tombstone)
# effective 1997 rather than inserting. Historical (pre-1997) orders still resolve the supplier
# via the as-of join; new orders after the delete find no open version -> unknown member (-1).
supp_del = spark.sql(f"SELECT s_suppkey AS supplier_id, s_name AS name, s_acctbal AS acctbal, s_comment AS comment FROM {sample_source_schema}.supplier WHERE s_suppkey = 3")
supp_del = with_op(with_effective(with_metadata(supp_del, "procurement", batch_id, load_ts), RUN3_SUPPLIER_EFFECTIVE), "D")
supp3 = supp3.unionByName(supp_del)
write_landing(supp3, "procurement", "supplier", INCREMENTAL_ZONE, load_ts)

ooo_addr = spark.sql(f"""
    SELECT c_custkey AS customer_id, 'BACKDATED CORRECTION (effective 1994)' AS address, c_nationkey AS nat_id
    FROM {sample_source_schema}.customer
    WHERE c_custkey = 1
""")
ooo_addr = with_effective(with_metadata(ooo_addr, "crm", batch_id, load_ts), RUN3_CORRECTION_EFFECTIVE)

# DQ demo (run 3): a malformed customer_address row lands alongside the correction. It violates a
# silver expectation (nation_key NOT NULL) and is routed to silver.customer_address_quarantine
# rather than the clean dimension (quarantineMode=table).
bad_addr3 = spark.sql("""
    SELECT CAST(910000002 AS BIGINT) AS customer_id, 'DQ_DEMO (run3): null nation_key (FK)' AS address, CAST(NULL AS BIGINT) AS nat_id
""")
bad_addr3 = with_effective(with_metadata(bad_addr3, "crm", batch_id, load_ts), RUN3_CORRECTION_EFFECTIVE)
ooo_addr = ooo_addr.unionByName(bad_addr3)
ooo_addr = with_op(ooo_addr)
write_landing(ooo_addr, "crm", "customer_address", INCREMENTAL_ZONE, load_ts)

# Late-arriving dimension master: part 9000001 finally arrives (it was referenced by a Run-2
# line item that resolved to the unknown member). From now on, new facts for 9000001 resolve to
# this real part; the Run-2 fact keeps the unknown member, since append-only facts are not
# retro-repointed. part is SCD2, so it carries the cdc_operation flag.
late_part = spark.sql("""
    SELECT CAST(9000001 AS BIGINT) AS p_partkey, 'LATE ARRIVAL WIDGET' AS p_name, 'Mfgr#9' AS p_mfgr,
           'Brand#99' AS p_brand, 'LATE ARRIVAL TYPE' AS p_type, CAST(1 AS INT) AS p_size,
           'SM BOX' AS p_container, CAST(999.99 AS DECIMAL(18,2)) AS p_retailprice,
           'late-arriving part master' AS p_comment
""")
late_part = with_op(with_metadata(late_part, "product_catalog", batch_id, load_ts))
write_landing(late_part, "product_catalog", "part", INCREMENTAL_ZONE, load_ts)

print("Run 3 supplier update + delete, backdated customer_address correction, late part master staged.")